<a href="https://colab.research.google.com/github/angelolucero-cloud/GO-Grupo-2/blob/main/Entrega%202/Parte%20A%20Formulaci%C3%B3n/Entrega-2-Parte-A-Formulacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Primer punto (1/8)
### 1. Balance de inventario por SKU y semana

Se define el balance para cada producto $i$ y semana $t$ del horizonte de planificación.

| Símbolo | Definición |
| :--- | :--- |
| $i$ | Índice del SKU. |
| $t$ | Semana del horizonte, desde 1 hasta 13. |
| $I_{it}$ | Inventario disponible al final de la semana. |
| $B_{it}$ | Pedidos pendientes al final de la semana. |
| $x_{it}$ | Producción antes de descontar la merma. |
| $d_{it}$ | Demanda del SKU durante la semana. |
| $\rho_{\ell(i)}$ | Rendimiento de la línea asignada al SKU. |

La producción útil es $\rho_{\ell(i)}x_{it}$. Por lo tanto, el balance es:

$$
I_{it}-B_{it}
=
I_{i,t-1}-B_{i,t-1}
+\rho_{\ell(i)}x_{it}
-d_{it}
$$

El inventario neto final es igual al inventario neto anterior, más la producción útil, menos la demanda semanal.

Para la primera semana se utilizan el inventario inicial $I_{i0}$ y los pedidos atrasados iniciales $B_{i0}$. Tanto el inventario como los pedidos atrasados son variables no negativas:

$$
I_{it}\geq 0,\qquad B_{it}\geq 0
$$

#### 1.3. Incorporación del rendimiento de producción

El rendimiento $\rho_{\ell(i)}$ representa la proporción de la producción que resulta utilizable en la línea asignada al SKU $i$.

Si la tasa de merma de esa línea es $m_{\ell(i)}$, entonces:

$$
\rho_{\ell(i)}=1-m_{\ell(i)}
$$

Como $x_{it}$ representa la producción antes de descontar la merma, la cantidad disponible para atender la demanda es:

$$
q_{it}=\rho_{\ell(i)}x_{it}
$$

Por ejemplo, con una merma del 2 %, el rendimiento es 0,98. Una producción de 1.000 cajas aporta 980 cajas utilizables al balance de inventario.

En el modelo se utilizarán las tasas de merma estimadas en la Entrega 1. El ejemplo anterior es únicamente ilustrativo.

La merma se incorpora una sola vez mediante el rendimiento aplicado a la producción; no se vuelve a descontar del inventario ni se utiliza para aumentar la demanda.

#### 1.4. Pedidos atrasados y su penalización

La demanda que no se atiende durante una semana se mantiene como un pedido pendiente para las semanas siguientes. Este tratamiento se denomina *backlogging*.

La variable $B_{it}$ representa la cantidad de cajas del SKU $i$ que permanecen pendientes al finalizar la semana $t$:

$$
B_{it}\geq 0
$$

Los atrasos no desaparecen al cambiar de semana: el balance de inventario los arrastra hasta que sean atendidos.

Para representar su costo, se define $p_i$ como la penalización por cada caja pendiente durante una semana. El costo total de los atrasos se incorpora a la función objetivo mediante:

$$
C_{\text{atrasos}}
=
\sum_i \sum_{t=1}^{13} p_i B_{it}
$$

Una caja que permanece pendiente durante varias semanas genera penalización en cada una de ellas. El modelo compara este costo con las demás alternativas de producción e inventario.

In [1]:
import io
import pandas as pd
from google.colab import files

archivos = files.upload()

nombre_archivo = next(iter(archivos))
tabla_traspaso = pd.read_csv(
    io.BytesIO(archivos[nombre_archivo])
)

print("Dimensiones:", tabla_traspaso.shape)
print("\nColumnas:")
print(tabla_traspaso.columns.tolist())

display(tabla_traspaso.head())

Saving tabla_traspaso_entrega2.csv to tabla_traspaso_entrega2.csv
Dimensiones: (117, 14)

Columnas:
['Fecha', 'Horizonte', 'SKU', 'linea_envasado', 'formato_L', 'pronostico_cajas', 'sigma_error_cajas', 'n_errores', 'rho_merma', 'factor_produccion_bruta', 'sobrellenado_mL_botella', 'sobrellenado_L_caja', 'litros_por_caja', 'litros_efectivos_caja']


,Fecha,Horizonte,SKU,linea_envasado,formato_L,pronostico_cajas,sigma_error_cajas,n_errores,rho_merma,factor_produccion_bruta,sobrellenado_mL_botella,sobrellenado_L_caja,litros_por_caja,litros_efectivos_caja
0,2026-09-28,1,CIT-L05,L1,0.5,1392.0,433.148090,8,0.019217,1.019594,5.72,0.06864,6.0,6.06864
1,2026-10-05,2,CIT-L05,L1,0.5,1753.0,261.129962,8,0.019217,1.019594,5.72,0.06864,6.0,6.06864
2,2026-10-12,3,CIT-L05,L1,0.5,2080.0,659.049099,8,0.019217,1.019594,5.72,0.06864,6.0,6.06864
3,2026-10-19,4,CIT-L05,L1,0.5,1740.0,10663.337087,8,0.019217,1.019594,5.72,0.06864,6.0,6.06864
4,2026-10-26,5,CIT-L05,L1,0.5,2042.0,912.031798,8,0.019217,1.019594,5.72,0.06864,6.0,6.06864


In [2]:
datos = tabla_traspaso.copy()

# Comprobar que no existan combinaciones SKU-semana repetidas.
assert not datos.duplicated(["SKU", "Horizonte"]).any(), (
    "Existen registros repetidos para un mismo SKU y semana."
)

# Comprobar que cada SKU tenga las 13 semanas.
semanas_esperadas = set(range(1, 14))

for sku, grupo in datos.groupby("SKU"):
    assert set(grupo["Horizonte"]) == semanas_esperadas, (
        f"Faltan semanas o existen semanas inesperadas para {sku}."
    )

assert datos["pronostico_cajas"].notna().all()
assert (datos["pronostico_cajas"] >= 0).all()
assert datos["rho_merma"].between(0, 1, inclusive="left").all()

# Convertir la tasa de merma en rendimiento.
datos["rendimiento"] = 1 - datos["rho_merma"]

# Conjuntos y parámetros del balance.
SKUS = sorted(datos["SKU"].unique())
SEMANAS = list(range(1, 14))

demanda = datos.set_index(
    ["SKU", "Horizonte"]
)["pronostico_cajas"].to_dict()

rendimiento = datos.set_index(
    ["SKU", "Horizonte"]
)["rendimiento"].to_dict()

print(f"Datos preparados: {len(SKUS)} SKU y {len(SEMANAS)} semanas.")

display(
    datos[
        ["linea_envasado", "rho_merma", "rendimiento"]
    ].drop_duplicates().sort_values("linea_envasado")
)

Datos preparados: 9 SKU y 13 semanas.


,linea_envasado,rho_merma,rendimiento
0,L1,0.019217,0.980783
13,L2,0.031146,0.968854


In [3]:
archivos_iniciales = files.upload()

nombre_inicial = next(iter(archivos_iniciales))
inventario_inicial = pd.read_csv(
    io.BytesIO(archivos_iniciales[nombre_inicial])
)

print("Dimensiones:", inventario_inicial.shape)
print("\nColumnas:")
print(inventario_inicial.columns.tolist())

display(inventario_inicial)

Saving inventario_inicial.csv to inventario_inicial.csv
Dimensiones: (9, 5)

Columnas:
['sku', 'inventario_inicial_cajas', 'nivel_servicio_objetivo', 'lead_time_jarabe_semanas', 'lote_minimo_cajas']


,sku,inventario_inicial_cajas,nivel_servicio_objetivo,lead_time_jarabe_semanas,lote_minimo_cajas
0,NEC-D05,1024,0.95,1,350
1,NEC-D15,3162,0.95,1,600
2,NEC-D30,339,0.95,1,140
3,CIT-L05,1217,0.95,1,350
4,CIT-L15,1282,0.95,1,430
5,CIT-L30,255,0.95,1,70
6,FUN-B05,2135,0.92,1,550
7,FUN-B15,1261,0.92,1,360
8,FUN-B30,155,0.92,1,30


In [4]:
assert not inventario_inicial["sku"].duplicated().any(), (
    "Hay SKU repetidos en el inventario inicial."
)

assert set(inventario_inicial["sku"]) == set(SKUS), (
    "Los SKU del inventario inicial y del pronóstico no coinciden."
)

parametros_sku = inventario_inicial.set_index("sku")

stock_inicial = parametros_sku["inventario_inicial_cajas"]

assert stock_inicial.notna().all()
assert (stock_inicial >= 0).all()

I0 = stock_inicial.to_dict()
B0 = {sku: 0.0 for sku in SKUS}
penalizacion_atraso = {sku: 1400.0 for sku in SKUS}

resumen_inicial = pd.DataFrame({
    "Inventario inicial": pd.Series(I0),
    "Atraso inicial": pd.Series(B0),
    "Penalización CLP/caja/semana": pd.Series(penalizacion_atraso)
})

display(resumen_inicial)

,Inventario inicial,Atraso inicial,Penalización CLP/caja/semana
CIT-L05,1217,0.0,1400.0
CIT-L15,1282,0.0,1400.0
CIT-L30,255,0.0,1400.0
FUN-B05,2135,0.0,1400.0
FUN-B15,1261,0.0,1400.0
FUN-B30,155,0.0,1400.0
NEC-D05,1024,0.0,1400.0
NEC-D15,3162,0.0,1400.0
NEC-D30,339,0.0,1400.0


In [5]:
%pip install -q pulp==2.9.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.7/17.7 MB 53.0 MB/s eta 0:00:00


In [6]:
import pulp as pl

modelo = pl.LpProblem("Planificacion_Bebidas_del_Maipo", pl.LpMinimize)

PARES = [(sku, t) for sku in SKUS for t in SEMANAS]

# Variables expresadas en cajas.
x = pl.LpVariable.dicts("produccion_bruta", PARES, lowBound=0)
I = pl.LpVariable.dicts("inventario", PARES, lowBound=0)
B = pl.LpVariable.dicts("atraso", PARES, lowBound=0)

# Balance por SKU y semana.
for k, sku in enumerate(SKUS):
    for t in SEMANAS:
        inventario_anterior = I0[sku] if t == 1 else I[(sku, t - 1)]
        atraso_anterior = B0[sku] if t == 1 else B[(sku, t - 1)]

        modelo += (
            I[(sku, t)] - B[(sku, t)]
            == inventario_anterior - atraso_anterior
            + rendimiento[(sku, t)] * x[(sku, t)]
            - demanda[(sku, t)],
            f"balance_{k}_{t}"
        )

# Expresión que incorporaremos a la función objetivo completa.
costo_atrasos = pl.lpSum(
    penalizacion_atraso[sku] * B[(sku, t)]
    for sku, t in PARES
)

print("Variables creadas:", len(modelo.variables()))
print("Balances incorporados:", len(modelo.constraints))

print("\nEjemplo de balance:")
print(next(iter(modelo.constraints.values())))

Variables creadas: 351
Balances incorporados: 117

Ejemplo de balance:
-atraso_('CIT_L05',_1) + inventario_('CIT_L05',_1) - 0.9807826423866719*produccion_bruta_('CIT_L05',_1) = -175.0


# Segundo punto (2/8)
### 2. Preparación y enlace con la producción

#### 2.1. Variable binaria de preparación

Se define $y_{it}$ para indicar si se habilita la producción del SKU $i$ durante la semana $t$.

| Valor | Interpretación |
| :--- | :--- |
| $y_{it}=1$ | Se prepara la producción del SKU durante esa semana. |
| $y_{it}=0$ | No se prepara y su producción debe ser cero. |

Su dominio es:

$$
y_{it}\in\{0,1\}
$$

Para vincular esta decisión con la producción se utiliza:

$$
x_{it}\leq M_{it}y_{it}
$$

La cota $M_{it}$ representa un límite superior válido para la producción bruta del SKU durante esa semana. Se deducirá a partir de las capacidades disponibles.

Si $y_{it}=0$, la restricción obliga a que $x_{it}=0$. Si $y_{it}=1$, permite producir hasta $M_{it}$; el lote mínimo se incorporará en el punto 6.

In [8]:
archivos_modelo = files.upload()

tablas_modelo = {}

for nombre, contenido in archivos_modelo.items():
    tabla = pd.read_csv(io.BytesIO(contenido))
    tablas_modelo[nombre] = tabla

    print(f"\nArchivo: {nombre}")
    print("Dimensiones:", tabla.shape)
    print("Columnas:", tabla.columns.tolist())
    display(tabla.head())

Saving maestro_sku.csv to maestro_sku.csv
Saving capacidad_horizonte.csv to capacidad_horizonte (1).csv

Archivo: maestro_sku.csv
Dimensiones: (9, 13)
Columnas: ['sku', 'familia', 'nombre_familia', 'sabor', 'formato_L', 'botellas_por_caja', 'litros_por_caja', 'linea_envasado', 'tasa_envasado_cajas_h', 'costo_variable_caja_clp', 'precio_lista_caja_clp', 'costo_inventario_caja_semana_clp', 'penalidad_atraso_caja_semana_clp']


,sku,familia,nombre_familia,sabor,formato_L,botellas_por_caja,litros_por_caja,linea_envasado,tasa_envasado_cajas_h,costo_variable_caja_clp,precio_lista_caja_clp,costo_inventario_caja_semana_clp,penalidad_atraso_caja_semana_clp
0,NEC-D05,NEC,Nectares con pulpa,Durazno,0.5,12,6.0,L1,180,3840,6620,95,1400
1,NEC-D15,NEC,Nectares con pulpa,Durazno,1.5,6,9.0,L2,210,5760,9930,95,1400
2,NEC-D30,NEC,Nectares con pulpa,Durazno,3.0,4,12.0,L2,130,7680,13240,95,1400
3,CIT-L05,CIT,Citricos gasificados,Limon,0.5,12,6.0,L1,180,2820,5320,95,1400
4,CIT-L15,CIT,Citricos gasificados,Limon,1.5,6,9.0,L2,210,4230,7980,95,1400



Archivo: capacidad_horizonte (1).csv
Dimensiones: (13, 14)
Columnas: ['t', 'fecha_lunes', 'horas_mezcla_regular', 'horas_mezcla_extra_max', 'tasa_mezcla_L_h', 'cip_mezcla_horas', 'horas_L1_regular', 'horas_L1_extra_max', 'horas_L2_regular', 'horas_L2_extra_max', 'costo_hora_extra_L2_clp', 'costo_hora_extra_mezcla_clp', 'mantencion_programada_h_L2', 'mantencion_programada_h_mezcla']


,t,fecha_lunes,horas_mezcla_regular,horas_mezcla_extra_max,tasa_mezcla_L_h,cip_mezcla_horas,horas_L1_regular,horas_L1_extra_max,horas_L2_regular,horas_L2_extra_max,costo_hora_extra_L2_clp,costo_hora_extra_mezcla_clp,mantencion_programada_h_L2,mantencion_programada_h_mezcla
0,1,2026-09-28,55.0,10,2300,1.2,40.0,8,48.0,16,168000,121000,0.0,0.0
1,2,2026-10-05,55.0,10,2300,1.2,40.0,8,48.0,16,168000,121000,0.0,0.0
2,3,2026-10-12,55.0,10,2300,1.2,40.0,8,48.0,16,168000,121000,0.0,0.0
3,4,2026-10-19,55.0,10,2300,1.2,40.0,8,40.0,16,168000,121000,8.0,0.0
4,5,2026-10-26,55.0,10,2300,1.2,40.0,8,48.0,16,168000,121000,0.0,0.0


#### 2.2. Cota superior de producción

Se construye una cota física a partir de la tasa de envasado y las horas disponibles en la línea asignada al SKU:

$$
M_{it}=v_i\left(H_{\ell(i),t}^{reg}+H_{\ell(i),t}^{extra,max}\right)
$$

| Símbolo | Definición |
| :--- | :--- |
| $v_i$ | Tasa de envasado del SKU, en cajas por hora. |
| $H_{\ell(i),t}^{reg}$ | Horas regulares disponibles en su línea y semana. |
| $H_{\ell(i),t}^{extra,max}$ | Máximo de horas extra permitido. |

La cota supone que toda la capacidad de la línea se dedica al SKU, sin descontar preparaciones ni producción de otros productos. Por ello, constituye un límite superior válido; las restricciones de capacidad compartida y preparación limitarán posteriormente la producción efectiva.

Como $x_{it}$ representa producción bruta, esta cota no se multiplica por el rendimiento.

In [9]:
maestro = next(
    tabla for tabla in tablas_modelo.values()
    if {"sku", "tasa_envasado_cajas_h"}.issubset(tabla.columns)
).set_index("sku")

capacidad = next(
    tabla for tabla in tablas_modelo.values()
    if {"t", "horas_L1_regular"}.issubset(tabla.columns)
).set_index("t")

assert maestro.index.is_unique
assert capacidad.index.is_unique
assert set(maestro.index) == set(SKUS)
assert set(capacidad.index) == set(SEMANAS)

M = {}

for sku, t in PARES:
    linea = maestro.loc[sku, "linea_envasado"]
    tasa = float(maestro.loc[sku, "tasa_envasado_cajas_h"])

    horas_regulares = capacidad.loc[t, f"horas_{linea}_regular"]
    horas_extra = capacidad.loc[t, f"horas_{linea}_extra_max"]

    M[(sku, t)] = tasa * (horas_regulares + horas_extra)

tabla_M = pd.DataFrame(
    [(sku, t, valor) for (sku, t), valor in M.items()],
    columns=["SKU", "Semana", "Cota_produccion_bruta"]
)

print("Cotas calculadas:", len(M))
display(tabla_M.pivot(
    index="SKU",
    columns="Semana",
    values="Cota_produccion_bruta"
))

Cotas calculadas: 117


Semana,1,2,3,4,5,6,7,8,9,10,11,12,13
SKU,,,,,,,,,,,,,
CIT-L05,8640.0,8640.0,8640.0,8640.0,8640.0,8640.0,8640.0,8640.0,8640.0,8640.0,8640.0,8640.0,7920.0
CIT-L15,13440.0,13440.0,13440.0,11760.0,13440.0,13440.0,13440.0,13440.0,11760.0,13440.0,13440.0,13440.0,11760.0
CIT-L30,8320.0,8320.0,8320.0,7280.0,8320.0,8320.0,8320.0,8320.0,7280.0,8320.0,8320.0,8320.0,7280.0
FUN-B05,8640.0,8640.0,8640.0,8640.0,8640.0,8640.0,8640.0,8640.0,8640.0,8640.0,8640.0,8640.0,7920.0
FUN-B15,13440.0,13440.0,13440.0,11760.0,13440.0,13440.0,13440.0,13440.0,11760.0,13440.0,13440.0,13440.0,11760.0
FUN-B30,8320.0,8320.0,8320.0,7280.0,8320.0,8320.0,8320.0,8320.0,7280.0,8320.0,8320.0,8320.0,7280.0
NEC-D05,8640.0,8640.0,8640.0,8640.0,8640.0,8640.0,8640.0,8640.0,8640.0,8640.0,8640.0,8640.0,7920.0
NEC-D15,13440.0,13440.0,13440.0,11760.0,13440.0,13440.0,13440.0,13440.0,11760.0,13440.0,13440.0,13440.0,11760.0
NEC-D30,8320.0,8320.0,8320.0,7280.0,8320.0,8320.0,8320.0,8320.0,7280.0,8320.0,8320.0,8320.0,7280.0


In [10]:
# Una variable binaria por SKU y semana.
y = pl.LpVariable.dicts(
    "preparacion",
    PARES,
    cat="Binary"
)

# Enlace entre preparación y producción.
for k, sku in enumerate(SKUS):
    for t in SEMANAS:
        modelo += (
            x[(sku, t)] <= M[(sku, t)] * y[(sku, t)],
            f"enlace_{k}_{t}"
        )

print("Variables binarias creadas:", len(y))
print("Variables totales:", len(modelo.variables()))
print("Restricciones totales:", len(modelo.constraints))

print("\nEjemplo de enlace:")
print(modelo.constraints["enlace_0_1"])

Variables binarias creadas: 117
Variables totales: 468
Restricciones totales: 234

Ejemplo de enlace:
-8640.0*preparacion_('CIT_L05',_1) + produccion_bruta_('CIT_L05',_1) <= -0.0


#### 2.3. Interpretación del enlace

Para CIT-L05 en la semana 1, la restricción es:

$$
x_{\text{CIT-L05},1}\leq 8640\,y_{\text{CIT-L05},1}
$$

| Preparación | Consecuencia |
| :--- | :--- |
| $y_{\text{CIT-L05},1}=0$ | La producción debe ser cero. |
| $y_{\text{CIT-L05},1}=1$ | La producción puede alcanzar como máximo 8.640 cajas brutas. |

Este límite corresponde a dedicar al SKU todas las horas disponibles de su línea. La producción efectiva también deberá cumplir las restricciones de capacidad compartida, preparación y mezcla.

### 3. Capacidad de envasado y mezcla

#### 3.1. Recursos y unidades de capacidad

La producción utiliza una mezcladora compartida y dos líneas de envasado. Cada recurso dispone de una capacidad semanal limitada.

| Recurso | Productos que procesa | Tasa de producción |
| :--- | :--- | :--- |
| Mezcladora | Todas las familias y formatos. | 2.300 litros por hora. |
| Línea L1 | SKU de 0,5 litros. | 180 cajas por hora. |
| Línea L2 | SKU de 1,5 litros. | 210 cajas por hora. |
| Línea L2 | SKU de 3,0 litros. | 130 cajas por hora. |

Las horas regulares y los máximos de horas extra se obtienen de `capacidad_horizonte.csv` para cada semana.

Para envasado, la producción bruta se convierte a horas mediante:

$$
\text{Horas de envasado}_{it}=\frac{x_{it}}{v_i}
$$

El consumo de jarabe asociado a esa producción se calcula incluyendo el sobrellenado:

$$
\text{Litros requeridos}_{it}=x_{it}L_i^{ef}
$$

Aquí, $L_i^{ef}$ representa los litros por caja, incluido el sobrellenado, y se obtiene de la tabla de traspaso de la Entrega 1.

Las preparaciones y limpiezas también consumen tiempo. Sus horas se incorporarán junto con las horas de procesamiento en las restricciones de capacidad.

El lead time de una semana del jarabe se incorporará en el punto 6 para vincular temporalmente la mezcla con el envasado.

#### 3.2. Capacidad semanal de envasado

Los SKU asignados a una misma línea comparten sus horas disponibles. Por ello, se suman las horas de producción y de preparación de todos sus productos:

$$
\sum_{i\in S_\ell}
\left(
\frac{x_{it}}{v_i}+s_i y_{it}
\right)
\leq
H_{\ell t}^{reg}+e_{\ell t}
\qquad \forall \ell,t
$$

| Símbolo | Definición |
| :--- | :--- |
| $S_\ell$ | Conjunto de SKU asignados a la línea $\ell$. |
| $x_{it}/v_i$ | Horas de envasado de la producción bruta. |
| $s_i$ | Tiempo de preparación estimado por lote activo del SKU. |
| $y_{it}$ | Variable binaria que activa la preparación. |
| $H_{\ell t}^{reg}$ | Horas regulares disponibles en la semana. |
| $e_{\ell t}$ | Horas extra utilizadas en la línea y semana. |

Las horas extra se limitan mediante:

$$
0\leq e_{\ell t}\leq H_{\ell t}^{extra,max}
$$

**Supuesto de preparación del modelo táctico**

Se utilizará un tiempo de preparación estimado por lote activo, que se determinará a partir de la matriz de limpiezas y de los cambios de formato.

Esta aproximación se documentará porque las limpiezas reales dependen del orden de producción. En la Entrega Final se contrastarán las horas estimadas con las que requiere el secuenciamiento.

L1 procesa únicamente el formato de 0,5 L, por lo que no se le asignarán cambios de formato entre sus SKU. L2 procesa dos formatos y deberá considerar esos cambios.

Las horas regulares del archivo de capacidad se interpretan como horas disponibles. Antes de programar las restricciones se verificará cómo están reflejadas las mantenciones para evitar descontarlas dos veces.

#### 3.3. Capacidad semanal de mezcla

La mezcladora es un recurso compartido por todas las familias. Su capacidad debe cubrir tanto la preparación del jarabe como las limpiezas CIP.

Se definen las siguientes variables:

| Símbolo | Definición |
| :--- | :--- |
| $V_{ft}$ | Litros de jarabe preparados para la familia $f$ durante la semana $t$. |
| $z_{ft}$ | Variable binaria que indica si se prepara jarabe de la familia $f$ en esa semana. |
| $e_{mt}$ | Horas extra utilizadas en la mezcladora durante la semana. |

La restricción de capacidad es:

$$
\frac{\sum_f V_{ft}}{r_{mt}}
+
c_t\sum_f z_{ft}
\leq
H_{mt}^{reg}+e_{mt}
\qquad \forall t
$$

Aquí, $r_{mt}$ es la tasa de mezcla en litros por hora y $c_t$ es el tiempo de CIP asignado por familia activa. Ambos parámetros se obtienen del archivo de capacidad.

Se adopta una aproximación táctica de una campaña por familia activa y semana, reservando un CIP de 1,2 horas por campaña. Esta reserva incluye la primera campaña y no representa la secuencia exacta de cambios entre familias.

Para activar la limpieza cuando se prepara jarabe, se utiliza:

$$
V_{ft}\leq U_t z_{ft}
$$

Una cota física válida para los litros preparados es:

$$
U_t=r_{mt}\left(H_{mt}^{reg}+H_{mt}^{extra,max}\right)
$$

Las horas extra de mezcla están acotadas por:

$$
0\leq e_{mt}\leq H_{mt}^{extra,max}
$$

**Consumo de jarabe y sobrellenado**

La necesidad de jarabe para envasar los SKU de una familia es:

$$
R_{ft}=\sum_{i\in S_f}L_i^{ef}x_{it}
$$

El parámetro $L_i^{ef}$ incluye los litros nominales por caja y el sobrellenado estimado en la Entrega 1.

Los litros preparados $V_{ft}$ y los litros consumidos $R_{ft}$ se vincularán mediante un balance de jarabe en el punto 6, respetando su lead time de una semana.

In [11]:
archivos_cip = files.upload()

nombre_cip = next(iter(archivos_cip))
setups_cip = pd.read_csv(
    io.BytesIO(archivos_cip[nombre_cip])
)

print("Columnas de setups_cip.csv:")
print(setups_cip.columns.tolist())

display(setups_cip)

print("\nHoras regulares y mantenciones por semana:")
display(capacidad.filter(regex="regular|mantencion"))

Saving setups_cip.csv to setups_cip.csv
Columnas de setups_cip.csv:
['familia_origen', 'familia_destino', 'horas_limpieza_cip', 'costo_cip_clp', 'requiere_cip']


,familia_origen,familia_destino,horas_limpieza_cip,costo_cip_clp,requiere_cip
0,CIT,CIT,0.0,0,0
1,CIT,FUN,1.0,74000,1
2,CIT,NEC,1.0,74000,1
3,FUN,CIT,2.4,177600,1
4,FUN,FUN,0.0,0,0
5,FUN,NEC,1.4,103600,1
6,NEC,CIT,2.8,207200,1
7,NEC,FUN,2.8,207200,1
8,NEC,NEC,0.0,0,0



Horas regulares y mantenciones por semana:


,horas_mezcla_regular,horas_L1_regular,horas_L2_regular,mantencion_programada_h_L2,mantencion_programada_h_mezcla
t,,,,,
1,55.0,40.0,48.0,0.0,0.0
2,55.0,40.0,48.0,0.0,0.0
3,55.0,40.0,48.0,0.0,0.0
4,55.0,40.0,40.0,8.0,0.0
5,55.0,40.0,48.0,0.0,0.0
6,49.0,40.0,48.0,0.0,6.0
7,55.0,40.0,48.0,0.0,0.0
8,55.0,40.0,48.0,0.0,0.0
9,55.0,40.0,40.0,8.0,0.0


#### 3.4. Parámetros de preparación y capacidad disponible

Las horas regulares del archivo de capacidad ya reflejan las mantenciones programadas, por lo que se utilizan directamente.

Para el modelo táctico se calcula el promedio de los seis tiempos de CIP entre familias distintas:

$$
\bar{s}_{CIP}
=
\frac{1+1+2,4+1,4+2,8+2,8}{6}
=
1,9\text{ horas}
$$

Se reserva este tiempo por cada lote activo de envasado. En L2 se agrega una reserva de 0,6 horas por lote para representar los cambios de formato:

$$
s_i=
\begin{cases}
1,9 & \text{si el SKU pertenece a L1}\\
2,5 & \text{si el SKU pertenece a L2}
\end{cases}
$$

Esta es una aproximación por lote: reserva preparación incluso para el primero de la semana y puede sobreestimar el tiempo cuando se agrupan productos de la misma familia y formato. Tampoco garantiza cubrir todas las secuencias posibles. Su precisión se evaluará mediante el secuenciamiento de la Entrega Final.

La limpieza de la mezcladora se trata por separado, utilizando el parámetro de CIP por familia activa del archivo de capacidad.

In [12]:
cambios_familia = setups_cip[
    setups_cip["familia_origen"] != setups_cip["familia_destino"]
]

cip_promedio = cambios_familia["horas_limpieza_cip"].mean()

linea_sku = maestro["linea_envasado"].to_dict()
familia_sku = maestro["familia"].to_dict()
tasa_envasado = maestro["tasa_envasado_cajas_h"].to_dict()

tiempo_preparacion = {
    sku: cip_promedio + (0.6 if linea_sku[sku] == "L2" else 0.0)
    for sku in SKUS
}

# Verificar que los litros por caja sean constantes para cada SKU.
assert (
    datos.groupby("SKU")["litros_efectivos_caja"]
    .nunique(dropna=False)
    .eq(1)
    .all()
)
assert datos["litros_efectivos_caja"].notna().all()
assert (datos["litros_efectivos_caja"] > 0).all()

litros_efectivos = (
    datos.drop_duplicates("SKU")
    .set_index("SKU")["litros_efectivos_caja"]
    .to_dict()
)

resumen_preparacion = pd.DataFrame({
    "Linea": pd.Series(linea_sku),
    "Preparacion_estimada_h": pd.Series(tiempo_preparacion),
    "Litros_por_caja_con_sobrellenado": pd.Series(litros_efectivos)
})

print(f"CIP promedio entre familias: {cip_promedio:.2f} horas")
display(resumen_preparacion)

CIP promedio entre familias: 1.90 horas


,Linea,Preparacion_estimada_h,Litros_por_caja_con_sobrellenado
CIT-L05,L1,1.9,6.068640
CIT-L15,L2,2.5,9.027798
CIT-L30,L2,2.5,12.033716
FUN-B05,L1,1.9,6.068640
FUN-B15,L2,2.5,9.027798
FUN-B30,L2,2.5,12.033716
NEC-D05,L1,1.9,6.068640
NEC-D15,L2,2.5,9.027798
NEC-D30,L2,2.5,12.033716


In [14]:
LINEAS = ["L1", "L2"]

sku_por_linea = {
    linea: [sku for sku in SKUS if linea_sku[sku] == linea]
    for linea in LINEAS
}

e_linea = {
    (linea, t): pl.LpVariable(
        f"extra_{linea}_{t}",
        lowBound=0,
        upBound=float(capacidad.loc[t, f"horas_{linea}_extra_max"])
    )
    for linea in LINEAS
    for t in SEMANAS
}

horas_envasado = {}

for linea in LINEAS:
    for t in SEMANAS:
        horas_envasado[(linea, t)] = pl.lpSum(
            (1.0 / float(tasa_envasado[sku])) * x[(sku, t)]
            + tiempo_preparacion[sku] * y[(sku, t)]
            for sku in sku_por_linea[linea]
        )

        modelo += (
            horas_envasado[(linea, t)]
            <= float(capacidad.loc[t, f"horas_{linea}_regular"])
            + e_linea[(linea, t)],
            f"capacidad_{linea}_{t}"
        )

print("Restricciones de envasado incorporadas: 26")

Restricciones de envasado incorporadas: 26


In [15]:
FAMILIAS = sorted(set(familia_sku.values()))
PARES_FAMILIA = [(f, t) for f in FAMILIAS for t in SEMANAS]

V = pl.LpVariable.dicts("jarabe_preparado", PARES_FAMILIA, lowBound=0)
z = pl.LpVariable.dicts("familia_activa_mezcla", PARES_FAMILIA, cat="Binary")

e_mezcla = {
    t: pl.LpVariable(
        f"extra_mezcla_{t}",
        lowBound=0,
        upBound=float(capacidad.loc[t, "horas_mezcla_extra_max"])
    )
    for t in SEMANAS
}

horas_mezcla = {}
U_mezcla = {}

for t in SEMANAS:
    tasa = float(capacidad.loc[t, "tasa_mezcla_L_h"])
    cip = float(capacidad.loc[t, "cip_mezcla_horas"])
    horas_reg = float(capacidad.loc[t, "horas_mezcla_regular"])
    extra_max = float(capacidad.loc[t, "horas_mezcla_extra_max"])

    U_mezcla[t] = tasa * (horas_reg + extra_max)

    horas_mezcla[t] = (
        pl.lpSum(V[(f, t)] for f in FAMILIAS) / tasa
        + cip * pl.lpSum(z[(f, t)] for f in FAMILIAS)
    )

    modelo += (
        horas_mezcla[t] <= horas_reg + e_mezcla[t],
        f"capacidad_mezcla_{t}"
    )

    for f in FAMILIAS:
        modelo += (
            V[(f, t)] <= U_mezcla[t] * z[(f, t)],
            f"activar_mezcla_{f}_{t}"
        )

# Consumo de jarabe asociado al envasado.
R = {
    (f, t): pl.lpSum(
        litros_efectivos[sku] * x[(sku, t)]
        for sku in SKUS if familia_sku[sku] == f
    )
    for f, t in PARES_FAMILIA
}

print("Variables totales:", len(modelo.variables()))
print("Restricciones totales:", len(modelo.constraints))

print("\nEjemplo de capacidad de mezcla:")
print(modelo.constraints["capacidad_mezcla_1"])

Variables totales: 585
Restricciones totales: 312

Ejemplo de capacidad de mezcla:
-extra_mezcla_1 + 1.2*familia_activa_mezcla_('CIT',_1) + 1.2*familia_activa_mezcla_('FUN',_1) + 1.2*familia_activa_mezcla_('NEC',_1) + 0.0004347826086956522*jarabe_preparado_('CIT',_1) + 0.0004347826086956522*jarabe_preparado_('FUN',_1) + 0.0004347826086956522*jarabe_preparado_('NEC',_1) <= 55.0


#### 3.5. Revisión de las restricciones de capacidad

Se incorporaron las siguientes restricciones:

| Restricción | Cantidad |
| :--- | ---: |
| Capacidad de envasado: 2 líneas × 13 semanas. | 26 |
| Capacidad de mezcla: 13 semanas. | 13 |
| Activación de mezcla: 3 familias × 13 semanas. | 39 |
| Total agregado en este punto. | 78 |

El modelo acumula 585 variables y 312 restricciones.

Las restricciones de envasado incluyen procesamiento y preparación. Las de mezcla incluyen los litros preparados y los CIP por familia activa. Las horas extra tienen límites superiores definidos por los datos.

La vinculación entre el jarabe preparado y su consumo en envasado se incorporará en el punto 6, respetando el lead time de una semana. La factibilidad del plan se comprobará al resolver el modelo completo.

### 4. Stock de seguridad

#### 4.1. Nivel de servicio e incertidumbre de la demanda

El stock de seguridad es una reserva destinada a cubrir desviaciones de la demanda respecto del pronóstico.

Para calcularlo se utilizan dos parámetros:

| Parámetro | Fuente | Interpretación |
| :--- | :--- | :--- |
| $\alpha_i$ | `inventario_inicial.csv` | Nivel de servicio objetivo del SKU. |
| $\hat{\sigma}_{i,h}$ | Tabla de traspaso de la Entrega 1. | Desviación estándar del error de pronóstico para el SKU y horizonte correspondiente. |

Los niveles de servicio proporcionados son:

| Familia | Nivel de servicio objetivo |
| :--- | ---: |
| CIT | 95 % |
| NEC | 95 % |
| FUN | 92 % |

Para formular la reserva se interpretará $\alpha_i$ como la probabilidad objetivo de cubrir la demanda del período considerado, bajo el supuesto de disponibilidad del abastecimiento planificado. Esta medida es distinta del porcentaje de cajas efectivamente entregadas.

Se utilizará una aproximación normal del error de pronóstico, suponiendo que está centrado en cero y que su desviación estándar está representada por $\hat{\sigma}_{i,h}$.

La desviación estándar se toma para cada semana pronosticada. No se multiplicará automáticamente por la raíz del horizonte: el horizonte del pronóstico no equivale a un período de demanda acumulada.

Antes de imponer restricciones se revisarán los valores calculados y su relación con la demanda y la capacidad disponible.

#### 4.2. Deducción del stock de seguridad

La demanda real se representa como el pronóstico más un error aleatorio:

$$
D_{it}=\hat{d}_{it}+\varepsilon_{it}
$$

Se supone que el error sigue aproximadamente una distribución normal con media cero:

$$
\varepsilon_{it}\sim N\left(0,\hat{\sigma}_{i,h}^{\,2}\right)
$$

Si se dispone del pronóstico más una reserva $SS_{it}$, la condición de cobertura es:

$$
P\left(D_{it}\leq\hat{d}_{it}+SS_{it}\right)\geq\alpha_i
$$

Al sustituir la expresión de la demanda y estandarizar el error:

$$
P\left(
\frac{\varepsilon_{it}}{\hat{\sigma}_{i,h}}
\leq
\frac{SS_{it}}{\hat{\sigma}_{i,h}}
\right)
\geq\alpha_i
$$

Por lo tanto, la reserva mínima bajo estos supuestos es:

$$
SS_{it}=\Phi^{-1}(\alpha_i)\,\hat{\sigma}_{i,h}
$$

La función $\Phi^{-1}$ entrega el cuantil de la distribución normal estándar:

| Nivel de servicio | Cuantil aproximado |
| :--- | ---: |
| 95 % | 1,645 |
| 92 % | 1,405 |

El stock de seguridad se redondeará hacia arriba a cajas completas:

$$
SS_{it}
=
\left\lceil
\Phi^{-1}(\alpha_i)\,\hat{\sigma}_{i,h}
\right\rceil
$$

Esta deducción corresponde a la cobertura de la demanda de una semana. Su aplicación depende de los supuestos sobre el error y de la disponibilidad oportuna del abastecimiento; no garantiza por sí sola el servicio del plan completo.

In [16]:
import numpy as np
from statistics import NormalDist

nivel_servicio = parametros_sku["nivel_servicio_objetivo"].to_dict()

assert all(0.5 < nivel_servicio[sku] < 1 for sku in SKUS)
assert datos["sigma_error_cajas"].notna().all()
assert np.isfinite(datos["sigma_error_cajas"]).all()
assert (datos["sigma_error_cajas"] >= 0).all()

z_servicio = {
    sku: NormalDist().inv_cdf(float(nivel_servicio[sku]))
    for sku in SKUS
}

tabla_SS = datos[
    ["SKU", "Horizonte", "pronostico_cajas", "sigma_error_cajas"]
].copy()

tabla_SS["nivel_servicio"] = tabla_SS["SKU"].map(nivel_servicio)
tabla_SS["z_servicio"] = tabla_SS["SKU"].map(z_servicio)

tabla_SS["stock_seguridad_cajas"] = np.ceil(
    tabla_SS["z_servicio"] * tabla_SS["sigma_error_cajas"]
).astype(int)

# Cociente definido solo cuando el pronóstico es positivo.
tabla_SS["SS_sobre_pronostico"] = (
    tabla_SS["stock_seguridad_cajas"]
    / tabla_SS["pronostico_cajas"].replace(0, np.nan)
)

SS = tabla_SS.set_index(
    ["SKU", "Horizonte"]
)["stock_seguridad_cajas"].to_dict()

print("Reservas calculadas:", len(SS))

print("\nResumen por SKU:")
display(
    tabla_SS.groupby("SKU").agg(
        nivel_servicio=("nivel_servicio", "first"),
        SS_minimo=("stock_seguridad_cajas", "min"),
        SS_promedio=("stock_seguridad_cajas", "mean"),
        SS_maximo=("stock_seguridad_cajas", "max")
    ).round(2)
)

print("\nMayores reservas en relación con el pronóstico:")
display(
    tabla_SS.sort_values(
        "SS_sobre_pronostico", ascending=False
    ).head(10)
)

Reservas calculadas: 117

Resumen por SKU:


,nivel_servicio,SS_minimo,SS_promedio,SS_maximo
SKU,,,,
CIT-L05,0.95,377,3410.15,17540
CIT-L15,0.95,626,975.38,1486
CIT-L30,0.95,89,153.38,188
FUN-B05,0.92,544,849.54,1074
FUN-B15,0.92,270,646.00,1007
FUN-B30,0.92,132,238.15,300
NEC-D05,0.95,247,668.92,1182
NEC-D15,0.95,797,1347.46,1931
NEC-D30,0.95,155,215.77,289



Mayores reservas en relación con el pronóstico:


,SKU,Horizonte,pronostico_cajas,sigma_error_cajas,nivel_servicio,z_servicio,stock_seguridad_cajas,SS_sobre_pronostico
3,CIT-L05,4,1740.000000,10663.337087,0.95,1.644854,17540,10.080460
8,CIT-L05,9,2326.000000,10587.860588,0.95,1.644854,17416,7.487532
86,NEC-D05,9,926.674202,709.392502,0.95,1.644854,1167,1.259342
68,FUN-B30,4,301.000000,210.920601,0.92,1.405072,297,0.986711
97,NEC-D15,7,1798.919911,1068.299246,0.95,1.644854,1758,0.977253
75,FUN-B30,11,324.000000,213.490749,0.92,1.405072,300,0.925926
81,NEC-D05,4,1357.004720,718.141688,0.95,1.644854,1182,0.871036
102,NEC-D15,12,2231.392517,1173.754245,0.95,1.644854,1931,0.865379
101,NEC-D15,11,2043.692530,1039.138120,0.95,1.644854,1710,0.836721
91,NEC-D15,1,1964.866895,991.425780,0.95,1.644854,1631,0.830082


#### 4.3. Revisión de las reservas calculadas

Se obtuvieron 117 valores de stock de seguridad. Destacan CIT-L05 en las semanas 4 y 9, con reservas de 17.540 y 17.416 cajas, respectivamente.

Estos resultados provienen de desviaciones estándar del error superiores a 10.000 cajas. Antes de incorporar las restricciones se revisará su cálculo en la Entrega 1 y la presencia de errores extremos.

Las estimaciones originales se conservan durante la revisión; no se aplican recortes arbitrarios.

#### Diagnóstico de las reservas extremas de CIT-L05

Las reservas elevadas de los horizontes 4 y 9 están dominadas por la venta de 32.916 cajas registrada el 29/12/2025. La misma observación aparece en ambos horizontes por la superposición de las ventanas de evaluación.

Al excluir únicamente esa fecha del cálculo, las reservas disminuyen de 17.540 a 1.181 cajas en el horizonte 4 y de 17.416 a 801 cajas en el horizonte 9. Cada estimación pasa de ocho a siete errores.

Esta comparación demuestra sensibilidad a una observación extrema, pero no prueba que el registro sea incorrecto. Se conservan las estimaciones originales como escenario base y se documenta la alternativa para analizar su impacto en la planificación.

#### 4.4. Incorporación de la reserva y tratamiento de su incumplimiento

Se define $u_{it}\geq 0$ como el déficit respecto del stock de seguridad, medido sobre el inventario neto:

$$
I_{it}-B_{it}+u_{it}\geq SS_{it}
$$

Si el inventario neto alcanza la reserva, el déficit puede ser cero. Si queda por debajo, $u_{it}$ mide la distancia hasta la meta.

Esta formulación mantiene el backlogging permitido y permite identificar reservas que no pueden alcanzarse con la capacidad disponible. El cumplimiento del nivel de servicio no queda garantizado por esta restricción flexible.

El déficit se penaliza en la función objetivo:

$$
C_{\text{déficit de reserva}}
=
\sum_i\sum_t q_i u_{it}
$$

El enunciado no entrega un costo específico para incumplir la reserva. Como supuesto de política inicial, se adopta $q_i=1.400$ CLP por caja y semana, utilizando la penalidad de atraso como referencia. Su efecto deberá analizarse mediante sensibilidad.

Como el déficit se mide sobre inventario neto, un pedido atrasado aumenta tanto el costo de backlogging como el déficit de reserva. Por tanto, mientras ambas penalizaciones estén activas, el costo marginal conjunto del atraso será de 2.800 CLP por caja y semana.

In [18]:
u_SS = pl.LpVariable.dicts(
    "deficit_reserva",
    PARES,
    lowBound=0
)

# Supuesto de política, documentado para su sensibilidad.
penalizacion_reserva = {sku: 1400.0 for sku in SKUS}

for k, sku in enumerate(SKUS):
    for t in SEMANAS:
        modelo += (
            I[(sku, t)] - B[(sku, t)] + u_SS[(sku, t)]
            >= float(SS[(sku, t)]),
            f"meta_reserva_{k}_{t}"
        )

costo_deficit_reserva = pl.lpSum(
    penalizacion_reserva[sku] * u_SS[(sku, t)]
    for sku, t in PARES
)

print("Metas de reserva incorporadas:", len(PARES))
print("Variables totales:", len(modelo.variables()))
print("Restricciones totales:", len(modelo.constraints))

Metas de reserva incorporadas: 117
Variables totales: 702
Restricciones totales: 429


### 5. Horas extra como decisión de producción

#### 5.1. Variables, límites y costos

Las horas extra permiten ampliar la capacidad semanal de los recursos. Sus variables ya se incorporaron a las restricciones de envasado y mezcla.

| Recurso | Variable | Límite |
| :--- | :--- | :--- |
| Línea L1 | $e_{L1,t}$ | Máximo semanal indicado en el archivo de capacidad. |
| Línea L2 | $e_{L2,t}$ | Máximo semanal indicado en el archivo de capacidad. |
| Mezcladora | $e_{mt}$ | Máximo semanal indicado en el archivo de capacidad. |

Para cada recurso $r$ y semana $t$:

$$
0\leq e_{rt}\leq H_{rt}^{extra,max}
$$

Su contribución al costo total es:

$$
C_{\text{horas extra}}
=
\sum_{t=1}^{13}
\left(
c_{L1,t}e_{L1,t}
+c_{L2,t}e_{L2,t}
+c_{m,t}e_{mt}
\right)
$$

El archivo proporciona un costo de 168.000 CLP por hora extra de L2 y 121.000 CLP por hora extra de mezcla.

No aparece un costo explícito para las horas extra de L1 en el enunciado ni en las columnas del archivo de capacidad. Este parámetro requiere un supuesto documentado antes de completar el costo de horas extra; no debe interpretarse como cero.

#### 5.2. Supuesto para el costo de horas extra de L1

Ante la ausencia de una tarifa específica para L1, se adopta como referencia inicial la tarifa de horas extra de L2.

Este supuesto permite valorar económicamente las horas adicionales de ambas líneas de envasado. No constituye un dato observado ni implica que sus costos reales sean iguales.

| Recurso | Costo por hora extra | Fuente |
| :--- | ---: | :--- |
| L1 | Tarifa de L2: 168.000 CLP en el escenario base. | Supuesto de referencia. |
| L2 | 168.000 CLP. | Archivo de capacidad. |
| Mezcladora | 121.000 CLP. | Archivo de capacidad. |

Se analizará la sensibilidad del plan a la tarifa asumida para L1.

In [19]:
# Supuesto: L1 utiliza inicialmente el 100 % de la tarifa de L2.
factor_tarifa_L1 = 1.0

costo_extra_linea = {}
costo_extra_mezcla = {}

for t in SEMANAS:
    tarifa_L2 = float(capacidad.loc[t, "costo_hora_extra_L2_clp"])

    costo_extra_linea[("L1", t)] = factor_tarifa_L1 * tarifa_L2
    costo_extra_linea[("L2", t)] = tarifa_L2

    costo_extra_mezcla[t] = float(
        capacidad.loc[t, "costo_hora_extra_mezcla_clp"]
    )

resumen_costos_extra = pd.DataFrame({
    "Semana": SEMANAS,
    "Tarifa_L1_CLP_h": [
        costo_extra_linea[("L1", t)] for t in SEMANAS
    ],
    "Tarifa_L2_CLP_h": [
        costo_extra_linea[("L2", t)] for t in SEMANAS
    ],
    "Tarifa_mezcla_CLP_h": [
        costo_extra_mezcla[t] for t in SEMANAS
    ]
})

display(resumen_costos_extra)

,Semana,Tarifa_L1_CLP_h,Tarifa_L2_CLP_h,Tarifa_mezcla_CLP_h
0,1,168000.0,168000.0,121000.0
1,2,168000.0,168000.0,121000.0
2,3,168000.0,168000.0,121000.0
3,4,168000.0,168000.0,121000.0
4,5,168000.0,168000.0,121000.0
5,6,168000.0,168000.0,121000.0
6,7,168000.0,168000.0,121000.0
7,8,168000.0,168000.0,121000.0
8,9,168000.0,168000.0,121000.0
9,10,168000.0,168000.0,121000.0


In [20]:
costo_extra_envasado = pl.lpSum(
    costo_extra_linea[(linea, t)] * e_linea[(linea, t)]
    for linea in LINEAS
    for t in SEMANAS
)

costo_extra_mezcla_total = pl.lpSum(
    costo_extra_mezcla[t] * e_mezcla[t]
    for t in SEMANAS
)

costo_horas_extra = (
    costo_extra_envasado + costo_extra_mezcla_total
)

print("Variables de horas extra valoradas:",
      len(e_linea) + len(e_mezcla))
print("Costo de horas extra preparado para la función objetivo.")

Variables de horas extra valoradas: 39
Costo de horas extra preparado para la función objetivo.


### 6. Lote mínimo y lead time del jarabe

#### 6.1. Lote mínimo por SKU

Cuando se prepara un SKU, la cantidad producida debe alcanzar su lote mínimo. Si no se prepara, su producción debe ser cero.

Se utiliza el parámetro $L_i^{min}$ del archivo `inventario_inicial.csv`.

En esta formulación se interpreta el lote mínimo como cajas de producción bruta, coherentemente con la definición de $x_{it}$.

$$
x_{it}\geq L_i^{min}y_{it}
$$

Junto con el enlace del punto 2:

$$
L_i^{min}y_{it}\leq x_{it}\leq M_{it}y_{it}
$$

| Preparación | Producción permitida |
| :--- | :--- |
| $y_{it}=0$ | Producción igual a cero. |
| $y_{it}=1$ | Producción entre el lote mínimo y la cota de capacidad. |

Esta restricción evita activar preparaciones con cantidades inferiores al lote mínimo.

#### 6.2. Lead time del jarabe

El jarabe requiere una semana de preparación. Por lo tanto, el jarabe preparado durante la semana $t$ estará disponible para envasar desde la semana $t+1$.

Definimos:

| Símbolo | Descripción |
|---|---|
| $V_{f,t}$ | Litros de jarabe de la familia $f$ preparados en la semana $t$. |
| $R_{f,t}$ | Litros de jarabe requeridos para envasar los productos de la familia $f$ en la semana $t$. |
| $J_{f,t}$ | Inventario de jarabe disponible al finalizar la semana $t$. |

El consumo de jarabe depende de la producción bruta y de los litros efectivos por caja, incluyendo el sobrellenado:

$$
R_{f,t}=\sum_{i\in f}\ell_i^{ef}\,x_{i,t}
$$

Para las semanas 2 a 13, el balance es:

$$
J_{f,t}=J_{f,t-1}+V_{f,t-1}-R_{f,t}
$$

Además:

$$
J_{f,t}\geq 0
$$

Así, el modelo solo puede envasar si dispone de jarabe preparado previamente.

**Condición inicial:** los archivos revisados informan inventario inicial de producto terminado, pero no existencias de jarabe ni preparaciones pendientes de la semana anterior. Para el escenario base, se supone que ambas cantidades son cero:

$$
J_{f,1}=0-R_{f,1}
$$

Como el inventario de jarabe no puede ser negativo, este supuesto impide envasar durante la primera semana. La demanda de esa semana se atiende con el inventario inicial de producto terminado y, si este no alcanza, se registra atraso.

Este supuesto deberá declararse al interpretar los resultados y podrá revisarse mediante un escenario con disponibilidad inicial de jarabe.

Se permite conservar jarabe entre semanas. Al no disponer de información adicional, no se incorporan límites de almacenamiento ni vencimiento.

In [21]:
# 1. Lotes mínimos: cajas de producción bruta
lote_minimo = parametros_sku["lote_minimo_cajas"].to_dict()

for k, sku in enumerate(SKUS):
    for t in SEMANAS:
        modelo += (
            x[sku, t] >= float(lote_minimo[sku]) * y[sku, t],
            f"lote_minimo_{k}_{t}"
        )

# 2. Inventario de jarabe disponible al cierre de cada semana
J = pl.LpVariable.dicts(
    "inventario_jarabe",
    PARES_FAMILIA,
    lowBound=0
)

# Supuestos iniciales, expresados en litros
jarabe_inicial = {f: 0.0 for f in FAMILIAS}
jarabe_pendiente_inicial = {f: 0.0 for f in FAMILIAS}

# 3. Balance con lead time de una semana
for f in FAMILIAS:
    for t in SEMANAS:
        if t == 1:
            disponible = (
                jarabe_inicial[f]
                + jarabe_pendiente_inicial[f]
            )
        else:
            disponible = J[f, t - 1] + V[f, t - 1]

        modelo += (
            J[f, t] == disponible - R[f, t],
            f"balance_jarabe_{f}_{t}"
        )

print("Restricciones de lote mínimo incorporadas:", len(PARES))
print("Balances de jarabe incorporados:", len(PARES_FAMILIA))
print("Variables del modelo:", len(modelo.variables()))
print("Restricciones del modelo:", len(modelo.constraints))

Restricciones de lote mínimo incorporadas: 117
Balances de jarabe incorporados: 39
Variables del modelo: 741
Restricciones del modelo: 585


In [22]:
revision_lotes = parametros_sku[
    ["lote_minimo_cajas", "lead_time_jarabe_semanas"]
].copy()

revision_lotes["menor_M_del_horizonte"] = [
    min(M[sku, t] for t in SEMANAS)
    for sku in revision_lotes.index
]

revision_lotes["lote_compatible_con_M"] = (
    revision_lotes["lote_minimo_cajas"]
    <= revision_lotes["menor_M_del_horizonte"]
)

display(revision_lotes)

assert (
    revision_lotes["lead_time_jarabe_semanas"] == 1
).all(), "Hay SKU cuyo lead time no es de una semana."

assert revision_lotes["lote_compatible_con_M"].all(), (
    "Hay lotes mínimos que superan el límite M."
)

print("Revisión completada: lotes compatibles con M y lead time de una semana.")

,lote_minimo_cajas,lead_time_jarabe_semanas,menor_M_del_horizonte,lote_compatible_con_M
sku,,,,
NEC-D05,350,1,7920.0,True
NEC-D15,600,1,11760.0,True
NEC-D30,140,1,7280.0,True
CIT-L05,350,1,7920.0,True
CIT-L15,430,1,11760.0,True
CIT-L30,70,1,7280.0,True
FUN-B05,550,1,7920.0,True
FUN-B15,360,1,11760.0,True
FUN-B30,30,1,7280.0,True


Revisión completada: lotes compatibles con M y lead time de una semana.


### 7. Inventario terminal y efecto de fin de horizonte

#### 7.1. ¿Por qué se necesita una condición terminal?

El modelo considera 13 semanas. Si solo se contabilizan los costos y la demanda de ese período, puede resultar conveniente terminar la última semana con existencias muy bajas, porque el modelo no observa las necesidades posteriores.

Este comportamiento se denomina efecto de fin de horizonte. En este caso resulta especialmente relevante porque el término del período de planificación antecede al pico de verano de las familias CIT y FUN.

Para evitar el agotamiento artificial de existencias, se establecerá una reserva mínima al finalizar la semana 13:

$$
I_{i,13}-B_{i,13}\geq T_i
$$

donde:

| Símbolo | Descripción |
|---|---|
| $I_{i,13}$ | Inventario de producto terminado del SKU $i$ al finalizar la semana 13. |
| $B_{i,13}$ | Demanda pendiente del SKU $i$ al finalizar la semana 13. |
| $T_i$ | Reserva terminal mínima definida para el SKU $i$. |

La restricción se aplica al inventario neto de atrasos, para evitar que existencias comprometidas con demanda pendiente se contabilicen como reserva disponible.

La reserva terminal es una condición obligatoria de cierre. Se diferencia de las metas semanales de stock de seguridad, cuyo incumplimiento se permite mediante una penalización.

Como la tabla de traspaso contiene pronósticos únicamente hasta la semana 13, la reserva deberá definirse mediante una regla explícita y evaluar su impacto. No se presentará como un pronóstico de semanas posteriores.

#### 7.2. Regla para definir la reserva terminal

Se utilizará el promedio de los pronósticos de las últimas cuatro semanas como referencia del consumo semanal al cierre del horizonte:

$$
\bar{d}_i=\frac{1}{4}\sum_{t=10}^{13}\hat{d}_{i,t}
$$

La reserva terminal se calculará mediante:

$$
T_i=\left\lceil c_i\,\bar{d}_i\right\rceil
$$

donde $c_i$ representa las semanas de cobertura exigidas:

| Familia | Cobertura terminal | Criterio |
|---|---:|---|
| CIT | 1 semana | Mayor protección antes del pico de verano. |
| FUN | 1 semana | Mayor protección antes del pico de verano. |
| NEC | 0,5 semanas | Reserva de continuidad operativa. |

Las cantidades se redondean hacia arriba y se expresan en cajas de producto terminado, por lo que no se ajustan nuevamente por merma.

**Supuesto de planificación:** estas coberturas son una política propuesta para el escenario base; no corresponden a valores entregados por el enunciado. El promedio de las semanas 10 a 13 es una referencia y no un pronóstico de la demanda posterior ni una estimación del aumento estacional.

La reserva terminal no se suma automáticamente al stock de seguridad: ambas condiciones actúan sobre el mismo inventario neto. La reserva terminal constituye un mínimo obligatorio al cierre, mientras que la meta de seguridad puede presentar un déficit penalizado.

Al resolver el modelo se comprobará si esta política es factible con las capacidades disponibles. Posteriormente se compararán sus resultados con un escenario sin reserva terminal y con otras coberturas.

In [23]:
# 1. Coberturas propuestas, expresadas en semanas
cobertura_terminal = {
    "CIT": 1.0,
    "FUN": 1.0,
    "NEC": 0.5
}

# 2. Promedio del pronóstico de las semanas 10 a 13
promedio_demanda_final = {
    sku: sum(float(demanda[sku, t]) for t in range(10, 14)) / 4.0
    for sku in SKUS
}

# 3. Reserva terminal, en cajas de producto terminado
T_terminal = {
    sku: int(np.ceil(
        cobertura_terminal[familia_sku[sku]]
        * promedio_demanda_final[sku]
    ))
    for sku in SKUS
}

# 4. Incorporación de las restricciones
for k, sku in enumerate(SKUS):
    modelo += (
        I[sku, 13] - B[sku, 13] >= T_terminal[sku],
        f"reserva_terminal_{k}"
    )

# 5. Resumen de la política
tabla_terminal = pd.DataFrame([
    {
        "SKU": sku,
        "Familia": familia_sku[sku],
        "Demanda_promedio_semanas_10_13": promedio_demanda_final[sku],
        "Cobertura_semanas": cobertura_terminal[familia_sku[sku]],
        "Reserva_terminal_cajas": T_terminal[sku]
    }
    for sku in SKUS
])

display(tabla_terminal.round(2))

print("Restricciones terminales incorporadas:", len(SKUS))
print("Variables del modelo:", len(modelo.variables()))
print("Restricciones del modelo:", len(modelo.constraints))

,SKU,Familia,Demanda_promedio_semanas_10_13,Cobertura_semanas,Reserva_terminal_cajas
0,CIT-L05,CIT,2544.50,1.0,2545
1,CIT-L15,CIT,3009.90,1.0,3010
2,CIT-L30,CIT,580.25,1.0,581
3,FUN-B05,FUN,2685.72,1.0,2686
4,FUN-B15,FUN,1808.52,1.0,1809
5,FUN-B30,FUN,81.00,1.0,81
6,NEC-D05,NEC,1208.01,0.5,605
7,NEC-D15,NEC,2068.83,0.5,1035
8,NEC-D30,NEC,485.92,0.5,243


Restricciones terminales incorporadas: 9
Variables del modelo: 741
Restricciones del modelo: 594


#### 7.3. Revisión de la condición terminal

Se incorporaron nueve restricciones de reserva terminal, una por SKU. Los mínimos exigidos al finalizar la semana 13 son:

| SKU | Reserva terminal (cajas) |
|---|---:|
| CIT-L05 | 2545 |
| CIT-L15 | 3010 |
| CIT-L30 | 581 |
| FUN-B05 | 2686 |
| FUN-B15 | 1809 |
| FUN-B30 | 81 |
| NEC-D05 | 605 |
| NEC-D15 | 1035 |
| NEC-D30 | 243 |

Estas cantidades son mínimos de inventario neto de atrasos, no resultados de producción ni inventarios obtenidos al resolver el modelo.

La revisión confirma que se aplicó la regla de cobertura propuesta. Su factibilidad y su efecto sobre los costos, la producción y las horas extra se comprobarán al resolver y comparar escenarios.

Esta política reduce el incentivo a agotar el producto terminado al cierre, pero no garantiza por sí sola la cobertura del pico de verano ni establece una reserva de jarabe para semanas posteriores.

### 8. Compromisos de producción de la semana focal

#### 8.1. Producción mínima comprometida

La semana 12, correspondiente al lunes 14 de diciembre de 2026, debe cumplir los compromisos establecidos en el archivo `compromisos_semana_focal.csv`.

Se define $C_i$ como la cantidad de cajas comprometidas para el SKU $i$ en esa semana.

Como $x_{i,12}$ representa producción bruta, se aplica el rendimiento para expresar la producción disponible como producto terminado:

$$
r_{i,12}\,x_{i,12}\geq C_i
$$

donde $r_{i,12}$ es el rendimiento del SKU después de considerar la merma.

**Criterio de interpretación:** los compromisos se consideran cajas de producto terminado. Esta interpretación se contrastará con las columnas y unidades del archivo.

La condición exige producir durante la semana 12. Por lo tanto, el inventario acumulado en semanas anteriores no sustituye esta obligación de producción.

Los compromisos se incorporan como un mínimo de producción y no se suman automáticamente al pronóstico de demanda, para evitar contabilizarlos dos veces. Cualquier indicación de que corresponden a demanda adicional deberá revisarse antes de modificar el balance.

In [24]:
coincidencias = [
    nombre for nombre in tablas_modelo
    if "compromisos_semana_focal" in nombre.lower()
]

if coincidencias:
    nombre_compromisos = coincidencias[0]
    compromisos = tablas_modelo[nombre_compromisos].copy()
else:
    print("Selecciona el archivo compromisos_semana_focal.csv")
    archivos_compromisos = files.upload()

    coincidencias = [
        nombre for nombre in archivos_compromisos
        if "compromisos_semana_focal" in nombre.lower()
    ]

    if len(coincidencias) != 1:
        raise ValueError(
            "Debes subir un único archivo de compromisos de la semana focal."
        )

    nombre_compromisos = coincidencias[0]
    compromisos = pd.read_csv(
        io.BytesIO(archivos_compromisos[nombre_compromisos]),
        sep=None,
        engine="python"
    )

    tablas_modelo[nombre_compromisos] = compromisos.copy()

print("Archivo:", nombre_compromisos)
print("Dimensiones:", compromisos.shape)
print("Columnas:", compromisos.columns.tolist())

with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(compromisos)

Selecciona el archivo compromisos_semana_focal.csv


Saving compromisos_semana_focal.csv to compromisos_semana_focal.csv
Archivo: compromisos_semana_focal.csv
Dimensiones: (15, 10)
Columnas: ['orden', 'sku', 'familia', 'linea_envasado', 'cliente', 'cantidad_comprometida_cajas', 'dia_comprometido', 'hora_comprometida', 'peso_prioridad', 'multa_atraso_clp_h']


,orden,sku,familia,linea_envasado,cliente,cantidad_comprometida_cajas,dia_comprometido,hora_comprometida,peso_prioridad,multa_atraso_clp_h
0,OP-001,NEC-D05,NEC,L1,Mayorista Sur,389,2,14:00,1,26000
1,OP-002,NEC-D05,NEC,L1,CD Santiago,509,5,18:00,3,78000
2,OP-003,NEC-D15,NEC,L2,Cadena Retail B,584,5,18:00,4,104000
3,OP-004,NEC-D15,NEC,L2,Cadena Retail A,736,3,18:00,5,130000
4,OP-005,NEC-D30,NEC,L2,CD Santiago,313,3,18:00,3,78000
5,OP-006,CIT-L05,CIT,L1,CD Santiago,347,4,18:00,3,78000
6,OP-007,CIT-L05,CIT,L1,Cadena Retail B,492,4,18:00,4,104000
7,OP-008,CIT-L15,CIT,L2,Mayorista Sur,472,3,18:00,1,26000
8,OP-009,CIT-L15,CIT,L2,Mayorista Sur,554,3,14:00,1,26000
9,OP-010,CIT-L30,CIT,L2,Cadena Retail B,157,4,10:00,4,104000


In [25]:
# 1. Validar los datos antes de agrupar
compromisos["cantidad_comprometida_cajas"] = pd.to_numeric(
    compromisos["cantidad_comprometida_cajas"],
    errors="raise"
)

assert compromisos["sku"].notna().all(), "Hay órdenes sin SKU."
assert compromisos["cantidad_comprometida_cajas"].notna().all(), (
    "Hay órdenes sin cantidad comprometida."
)
assert (compromisos["cantidad_comprometida_cajas"] >= 0).all(), (
    "Hay cantidades comprometidas negativas."
)

sku_desconocidos = set(compromisos["sku"]) - set(SKUS)
assert not sku_desconocidos, (
    f"Hay SKU que no existen en el modelo: {sku_desconocidos}"
)

# 2. Sumar todas las órdenes de cada SKU
C_compromisos = (
    compromisos.groupby("sku")["cantidad_comprometida_cajas"]
    .sum()
    .to_dict()
)

# 3. Exigir producción terminada suficiente en la semana 12
for k, sku in enumerate(SKUS):
    if sku in C_compromisos:
        modelo += (
            rendimiento[sku, 12] * x[sku, 12]
            >= float(C_compromisos[sku]),
            f"compromiso_semana_12_{k}"
        )

# 4. Mostrar las cantidades comprometidas
tabla_compromisos = pd.DataFrame([
    {
        "SKU": sku,
        "Compromiso_producto_terminado_cajas": cantidad,
        "Produccion_bruta_minima_cajas": (
            float(cantidad) / float(rendimiento[sku, 12])
        )
    }
    for sku, cantidad in sorted(C_compromisos.items())
])

display(tabla_compromisos.round(2))

print("Órdenes consideradas:", len(compromisos))
print("Restricciones de compromisos incorporadas:", len(C_compromisos))
print("Variables del modelo:", len(modelo.variables()))
print("Restricciones del modelo:", len(modelo.constraints))

,SKU,Compromiso_producto_terminado_cajas,Produccion_bruta_minima_cajas
0,CIT-L05,839,855.44
1,CIT-L15,1026,1058.98
2,CIT-L30,157,162.05
3,FUN-B05,1340,1366.26
4,FUN-B15,860,887.65
5,FUN-B30,66,68.12
6,NEC-D05,898,915.60
7,NEC-D15,1320,1362.43
8,NEC-D30,313,323.06


Órdenes consideradas: 15
Restricciones de compromisos incorporadas: 9
Variables del modelo: 741
Restricciones del modelo: 603


#### 8.2. Revisión de los compromisos incorporados

Las 15 órdenes del archivo se agruparon por SKU, generando nueve restricciones de producción mínima para la semana 12.

| SKU | Compromiso total de producto terminado (cajas) |
|---|---:|
| CIT-L05 | 839 |
| CIT-L15 | 1026 |
| CIT-L30 | 157 |
| FUN-B05 | 1340 |
| FUN-B15 | 860 |
| FUN-B30 | 66 |
| NEC-D05 | 898 |
| NEC-D15 | 1320 |
| NEC-D30 | 313 |

Las restricciones consideran el rendimiento de cada SKU para convertir la producción bruta en cajas de producto terminado.

El modelo semanal exige cubrir estas cantidades mediante producción de la semana 12. El cumplimiento de los días y horas comprometidos deberá verificarse posteriormente mediante la secuenciación detallada.

La incorporación de las restricciones no demuestra todavía su factibilidad conjunta con las capacidades y las demás condiciones del modelo.

### 9. Función objetivo

El modelo busca minimizar el costo total de planificación durante las 13 semanas:

$$
\min Z =
C_{\text{producción}}
+ C_{\text{inventario}}
+ C_{\text{atrasos}}
+ C_{\text{preparaciones}}
+ C_{\text{horas extra}}
+ C_{\text{déficit de reserva}}
$$

| Componente | Forma de cálculo |
|---|---|
| Producción | Costo variable por caja multiplicado por la producción bruta de cada SKU. |
| Inventario | Costo por caja y semana multiplicado por el inventario final de producto terminado. |
| Atrasos | Penalización por caja y semana multiplicada por la demanda pendiente al cierre. |
| Preparaciones | Costos aproximados de limpieza CIP y cambios de formato asociados a las campañas activadas. |
| Horas extra | Horas extra utilizadas en envasado y mezcla, multiplicadas por sus respectivas tarifas. |
| Déficit de reserva | Penalización por cada caja faltante respecto de la meta semanal de inventario neto. |

#### Criterios de valoración

- El costo variable se aplica a la producción bruta. Se interpreta que los recursos consumidos en unidades que se pierden por merma también generan costos.
- El costo de producción se contabiliza una sola vez: no se añade nuevamente un costo de materiales al volumen de jarabe preparado.
- Los costos de preparación deben ser coherentes con las aproximaciones utilizadas en las restricciones de capacidad. No representan una secuencia exacta de cambios entre productos.
- Para L1 se conserva el supuesto de usar la tarifa de horas extra de L2, dado que no se dispone de una tarifa específica.
- Se mantiene la penalización propuesta de 1400 CLP por caja y semana de déficit de reserva. Es un supuesto de política, adicional a la penalización por atraso.
- Cuando un atraso incrementa también el déficit de reserva, ambos conceptos generan penalización.
- No se incorpora un costo de almacenamiento de jarabe, porque no se dispone de una tarifa. Esto constituye una limitación del modelo.
- Las reservas terminales y los compromisos de la semana 12 son restricciones obligatorias; no se permite sustituir su cumplimiento por una penalización.

Los resultados se interpretarán considerando estos supuestos y las sensibilidades correspondientes.

#### 9.1. Valoración aproximada de las preparaciones

En envasado se valoran 1,9 horas de CIP por SKU activo, a 74.000 CLP por hora. Para los SKU de L2 se añade una provisión de 38.000 CLP por cambio de formato por campaña activada.

En mezcla se consideran las 1,2 horas de CIP por familia activa incluidas en la capacidad semanal. Se utiliza como supuesto la misma tarifa de 74.000 CLP por hora.

Estas provisiones son aproximaciones de planificación: incluyen la primera campaña y no reproducen los cambios de una secuencia concreta. Su precisión deberá evaluarse frente a la programación detallada.

In [26]:
# 1. Costos unitarios del maestro
costo_variable = maestro["costo_variable_caja_clp"].to_dict()
costo_mantencion = (
    maestro["costo_inventario_caja_semana_clp"].to_dict()
)

costo_produccion = pl.lpSum(
    float(costo_variable[sku]) * x[sku, t]
    for sku, t in PARES
)

costo_inventario = pl.lpSum(
    float(costo_mantencion[sku]) * I[sku, t]
    for sku, t in PARES
)

# 2. Preparaciones aproximadas de envasado
tarifa_cip_envasado = 74000.0
costo_cambio_formato = 38000.0

costo_preparacion_sku = {
    sku: (
        float(cip_promedio) * tarifa_cip_envasado
        + (
            costo_cambio_formato
            if linea_sku[sku] == "L2"
            else 0.0
        )
    )
    for sku in SKUS
}

costo_preparaciones_envasado = pl.lpSum(
    costo_preparacion_sku[sku] * y[sku, t]
    for sku, t in PARES
)

# 3. CIP de mezcla: tarifa asumida
tarifa_cip_mezcla = 74000.0

costo_preparaciones_mezcla = pl.lpSum(
    tarifa_cip_mezcla
    * float(capacidad.loc[t, "cip_mezcla_horas"])
    * z[f, t]
    for f, t in PARES_FAMILIA
)

costo_preparaciones = (
    costo_preparaciones_envasado
    + costo_preparaciones_mezcla
)

# 4. Reunir todos los componentes
componentes_costo = {
    "Producción": costo_produccion,
    "Inventario": costo_inventario,
    "Atrasos": costo_atrasos,
    "Preparaciones de envasado": costo_preparaciones_envasado,
    "Preparaciones de mezcla": costo_preparaciones_mezcla,
    "Horas extra": costo_horas_extra,
    "Déficit de reserva": costo_deficit_reserva
}

costo_total = pl.lpSum(componentes_costo.values())

# Establecer la función objetivo sin modificar las restricciones
modelo.setObjective(costo_total)

print("Función objetivo incorporada: minimizar el costo total.")
print("Componentes incluidos:")
for nombre in componentes_costo:
    print("-", nombre)

print("Variables del modelo:", len(modelo.variables()))
print("Restricciones del modelo:", len(modelo.constraints))
print("El modelo todavía no se ha resuelto.")

Función objetivo incorporada: minimizar el costo total.
Componentes incluidos:
- Producción
- Inventario
- Atrasos
- Preparaciones de envasado
- Preparaciones de mezcla
- Horas extra
- Déficit de reserva
Variables del modelo: 741
Restricciones del modelo: 603
El modelo todavía no se ha resuelto.


In [27]:
import math

# 1. Verificar la función objetivo
assert modelo.objective is not None, "Falta la función objetivo."
assert modelo.sense == pl.LpMinimize, (
    "El modelo debe minimizar costos."
)

assert all(
    math.isfinite(float(coeficiente))
    for coeficiente in modelo.objective.values()
), "La función objetivo contiene coeficientes no válidos."

# 2. Revisar los grupos de restricciones
grupos_esperados = {
    "Balances de producto terminado": ("balance_", 117),
    "Enlace entre producción y setup": ("enlace_", 117),
    "Capacidad de L1": ("capacidad_L1_", 13),
    "Capacidad de L2": ("capacidad_L2_", 13),
    "Capacidad de mezcla": ("capacidad_mezcla_", 13),
    "Activación de familias en mezcla": ("activar_mezcla_", 39),
    "Metas de stock de seguridad": ("meta_reserva_", 117),
    "Lotes mínimos": ("lote_minimo_", 117),
    "Balances de jarabe": ("balance_jarabe_", 39),
    "Reservas terminales": ("reserva_terminal_", 9),
    "Compromisos de la semana 12": ("compromiso_semana_12_", 9)
}

revision_estructura = []

for grupo, (prefijo, esperado) in grupos_esperados.items():
    # Evitar contar balances de jarabe como producto terminado
    encontrados = sum(
        nombre.startswith(prefijo)
        and not (
            prefijo == "balance_"
            and nombre.startswith("balance_jarabe_")
        )
        for nombre in modelo.constraints
    )

    revision_estructura.append({
        "Grupo": grupo,
        "Esperadas": esperado,
        "Encontradas": encontrados,
        "Correcto": encontrados == esperado
    })

revision_estructura = pd.DataFrame(revision_estructura)
display(revision_estructura)

assert revision_estructura["Correcto"].all(), (
    "Hay diferencias en los grupos de restricciones."
)

# 3. Verificar coeficientes y constantes de las restricciones
for nombre, restriccion in modelo.constraints.items():
    assert math.isfinite(float(restriccion.constant)), (
        f"Constante no válida en {nombre}"
    )
    assert all(
        math.isfinite(float(coeficiente))
        for coeficiente in restriccion.values()
    ), f"Coeficiente no válido en {nombre}"

print("Revisión estructural completada.")
print("Variables:", len(modelo.variables()))
print("Restricciones:", len(modelo.constraints))
print("Componentes de costo:", len(componentes_costo))
print("La factibilidad y la solución se evaluarán en la parte B.")

,Grupo,Esperadas,Encontradas,Correcto
0,Balances de producto terminado,117,117,True
1,Enlace entre producción y setup,117,117,True
2,Capacidad de L1,13,13,True
3,Capacidad de L2,13,13,True
4,Capacidad de mezcla,13,13,True
5,Activación de familias en mezcla,39,39,True
6,Metas de stock de seguridad,117,117,True
7,Lotes mínimos,117,117,True
8,Balances de jarabe,39,39,True
9,Reservas terminales,9,9,True


Revisión estructural completada.
Variables: 741
Restricciones: 603
Componentes de costo: 7
La factibilidad y la solución se evaluarán en la parte B.
